# Credit-Lens: Análise Exploratória de Dados (EDA)

Este notebook documenta as descobertas centrais que orientaram as decisões de modelagem de risco de crédito no pipeline:
1. **Distribuição do Target**: Desbalanceamento de classe (~8% inadimplentes).
2. **Tratamento do Valor Sentinela**: Anomalia em `DAYS_EMPLOYED = 365243` convertida para NaN com flag indicadora.
3. **Agregações Comportamentais**: Importância dos dados históricos de bureau e parcelas anteriores.
4. **Assimetria Financeira**: Concentração em renda declarada e valor de crédito.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Caminho para o dataset processado via DuckDB
PROCESSED = Path("../data/processed/dataset.parquet")
df = pd.read_parquet(PROCESSED)
print(f"Shape do dataset: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

## 1. Distribuição da Variável Alvo (`TARGET`)

O problema é fortemente desbalanceado, com taxa de inadimplência típica de carteiras de varejo.

In [ ]:
target_dist = df["TARGET"].value_counts(normalize=True).rename({0: "Adimplente (0)", 1: "Inadimplente (1)"})
print("Distribuição percentual do Target:")
print(target_dist.map("{:.2%}".format))

taxa_inadimplencia = df["TARGET"].mean()
print(f"\nTaxa média de inadimplência: {taxa_inadimplencia:.2%}")

## 2. Diagnóstico da Anomalia em `DAYS_EMPLOYED`

No dataset original do Home Credit, o valor `365243` (~1000 anos) indica clientes aposentados ou sem emprego formal registrado. No pipeline de features, criamos a flag binária `DAYS_EMPLOYED_ANOMALO` e substituímos o valor contínuo por `NaN`.

In [ ]:
anomalos = df["DAYS_EMPLOYED_ANOMALO"].value_counts()
inadimp_por_anomalo = df.groupby("DAYS_EMPLOYED_ANOMALO")["TARGET"].mean()

print("Contagem de clientes com valor anômalo em DAYS_EMPLOYED:")
print(anomalos)
print("\nTaxa de inadimplência por grupo (0=Emprego normal, 1=Aposentado/Sem vínculo):")
print(inadimp_por_anomalo.map("{:.2%}".format))

## 3. Comportamento Histórico (Bureau e Parcelas)

Análise da relação entre histórico em outros birôs (`bureau`) e adimplência no contrato atual.

In [ ]:
cols_analise = [
    "bureau_n_creditos",
    "bureau_atraso_max_dias",
    "prev_n_recusados",
    "inst_atraso_medio_dias",
    "inst_taxa_em_dia"
]
df.groupby("TARGET")[cols_analise].median()

## 4. Conclusões e Decisões de Modelagem

- **Métrica de corte**: Devido ao desbalanceamento (8%), a acurácia é inadequada. Focamos em **ROC-AUC**, **KS** e **Gini**.
- **Ponderação de Classes**: Uso de `class_weight='balanced'` nos modelos.
- **Features Comportamentais**: O histórico consolidado de parcelas e recusas anteriores tem correlação direta com a taxa de default.